In [ ]:
# Figure 6a: Predicted and original molecules
import json
import random
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, DataStructs

def tanimoto_from_mols(mol_a, mol_b, radius=2, n_bits=2048):
    if mol_a is None or mol_b is None:
        return None
    fp_a = AllChem.GetMorganFingerprintAsBitVect(mol_a, radius, nBits=n_bits)
    fp_b = AllChem.GetMorganFingerprintAsBitVect(mol_b, radius, nBits=n_bits)
    return DataStructs.TanimotoSimilarity(fp_a, fp_b)

with open('../models/contrastive0.5/outputs/test_outputs_1_attempts.json', 'r') as f:
    data = json.load(f)

# Select 10 random samples where the predicted SMILES string is not equal to the original SMILES string
wrong_samples = [sample for sample in data if sample['predicted'][0] != sample['original']]
random_samples = random.sample(wrong_samples, 5)

pred_mols = []
orig_mols = []
pred_legends = []
orig_legends = []

for sample in random_samples:
    pred_smiles, orig_smiles = sample['predicted'][0], sample['original']
    pred_mol, orig_mol = Chem.MolFromSmiles(pred_smiles), Chem.MolFromSmiles(orig_smiles)
    sim = tanimoto_from_mols(pred_mol, orig_mol)
    sim_text = f"Tanimoto={sim:.2f}"
    pred_mols.append(pred_mol)
    orig_mols.append(orig_mol)
    pred_iupac = pcp.get_compounds(pred_smiles, 'smiles')
    orig_iupac = pcp.get_compounds(orig_smiles, 'smiles')
    
    pred_legends.append(f"{pred_smiles}\n{pred_iupac[0].iupac_name}\n{sim_text}")
    orig_legends.append(f"{orig_smiles}\n{orig_iupac[0].iupac_name}")

# Draw grids
img_pred = Draw.MolsToGridImage(
    pred_mols,
    molsPerRow=5,
    subImgSize=(300,300),
    legends=pred_legends
)

img_gt = Draw.MolsToGridImage(
    orig_mols,
    molsPerRow=5,
    subImgSize=(300,300),
    legends=orig_legends
)

display(img_pred)
display(img_gt)

In [ ]:
# Figure 6b: Distribution of Tanimoto similarities for predicted vs original molecules
import numpy as np
import matplotlib.pyplot as plt

tanimoto_scores = []
for sample in data:
    pred_smiles = sample['predicted'][0]
    orig_smiles = sample['original']
    pred_mol, orig_mol = Chem.MolFromSmiles(pred_smiles), Chem.MolFromSmiles(orig_smiles)
    sim = tanimoto_from_mols(pred_mol, orig_mol)
    if sim is not None:
        tanimoto_scores.append(sim)

tanimoto_scores = np.array(tanimoto_scores)

print(f"Mean similarity: {np.mean(tanimoto_scores):.3f}")
print(f"Total evaluated pairs: {len(tanimoto_scores)}")

# Plot histogram
plt.figure(figsize=(5, 3.5), dpi=300)
plt.hist(tanimoto_scores, bins=30, edgecolor='black')
plt.xlabel('Tanimoto similarity')
plt.ylabel('Count')
plt.title('Distribution of Tanimoto Similarities')
plt.tight_layout()
plt.show()